In [10]:
import io
import re
import html
import random
from dataclasses import dataclass, asdict
from typing import Dict, List, Optional, Any

import pandas as pd
import sympy as sp
from sympy.parsing.latex import parse_latex

# Optional: only needed in Jupyter notebooks
from IPython.display import display, Math


# ============================================================
# 1. DATA STRUCTURES
# ============================================================

@dataclass
class QuestionData:
    """Stores all parsed information from one question file."""
    topic: str = ""
    question_id: str = ""
    question_type: Optional[int] = None

    question_en: str = ""
    question_zh: str = ""

    mark: Optional[int] = None
    mc_choice: Optional[pd.DataFrame] = None

    hint_en: str = ""
    hint_zh: str = ""

    variables: Optional[pd.DataFrame] = None
    variable_values: Optional[Dict[str, int]] = None

    answers_raw: Optional[List[str]] = None
    answers_filled: Optional[List[str]] = None
    answers_with_final_unit: Optional[List[str]] = None

    unit: str = ""
    remark: str = ""

    explanation_en: str = ""
    explanation_zh: str = ""

    image_path: str = ""
    image_pos: Optional[int] = None

    difficulty: Optional[int] = None
    trash: Optional[int] = None


# ============================================================
# 2. BASIC TEXT UTILITIES
# ============================================================

def normalize_text(text: str) -> str:
    """
    Convert HTML escaped characters such as:
        &lt;  -> <
        &gt;  -> >
    so the parser can work with the file content.
    """
    return html.unescape(text)


def clean_math_text(text: str) -> str:
    """
    Clean LaTeX/math text a little so SymPy can parse it more reliably.
    """
    if not text:
        return text

    # Replace special minus signs with normal minus
    text = text.replace("–", "-").replace("−", "-")

    # Remove surrounding whitespace
    return text.strip()


def to_int_or_none(value: str) -> Optional[int]:
    """
    Convert a string to int safely.
    Return None if conversion is not possible.
    """
    value = value.strip()
    if value == "":
        return None
    try:
        return int(value)
    except ValueError:
        return None


# ============================================================
# 3. TAG EXTRACTION
# ============================================================

def extract_tag(text: str, tag_name: str) -> str:
    """
    Extract content between:
        <\\begin{tag_name}>
        <\\end{tag_name}>

    Returns empty string if tag is not found.
    """
    pattern = rf"<\\begin\{{{re.escape(tag_name)}\}}>(.*?)<\\end\{{{re.escape(tag_name)}\}}>"
    match = re.search(pattern, text, flags=re.DOTALL)
    return match.group(1).strip() if match else ""


def extract_all_tags(text: str) -> Dict[str, str]:
    """
    Extract all tags from the file into a dictionary:
        { tag_name: tag_content, ... }
    """
    pattern = r"<\\begin\{(.*?)\}>(.*?)<\\end\{\1\}>"
    matches = re.findall(pattern, text, flags=re.DOTALL)

    result = {}
    for tag_name, tag_content in matches:
        result[tag_name.strip()] = tag_content.strip()

    return result


# ============================================================
# 4. BLOCK PARSERS
# ============================================================

def parse_csv_block(raw: str) -> pd.DataFrame:
    """
    Parse blocks like:

        [
            Variable,liminf,limsup
            *t_v*,22,30
            *m_v*,50,200
        ]

    into a pandas DataFrame.
    """
    if not raw.strip():
        return pd.DataFrame()

    # Remove outer square brackets if present
    raw = raw.strip()
    if raw.startswith("[") and raw.endswith("]"):
        raw = raw[1:-1]

    # Keep only meaningful lines
    lines = []
    for line in raw.splitlines():
        line = line.strip().replace("\t", "")
        if line and "," in line:
            lines.append(line)

    if not lines:
        return pd.DataFrame()

    csv_content = "\n".join(lines)

    return pd.read_csv(
        io.StringIO(csv_content),
        sep=",",
        skipinitialspace=True,
        engine="python"
    )


def parse_list_block(raw: str) -> List[str]:
    """
    Parse blocks like:

        [
            line 1,
            line 2,
            line 3
        ]

    into a Python list of strings.

    This is used for the Answer block.
    """
    if not raw.strip():
        return []

    raw = raw.strip()

    # Remove outer [ ... ] if present
    if raw.startswith("[") and raw.endswith("]"):
        raw = raw[1:-1]

    items = []
    for line in raw.splitlines():
        line = line.strip()

        # Skip empty lines
        if not line:
            continue

        # Remove trailing comma if present
        if line.endswith(","):
            line = line[:-1].strip()

        if line:
            items.append(line)

    return items


# ============================================================
# 5. VARIABLE HANDLING
# ============================================================

def sample_variables(variables_df: pd.DataFrame, rng: Optional[random.Random] = None) -> Dict[str, int]:
    """
    From a Variable table:
        Variable, liminf, limsup
    randomly generate values for each variable.
    """
    if rng is None:
        rng = random.Random()

    values = {}

    if variables_df is None or variables_df.empty:
        return values

    required_cols = {"Variable", "liminf", "limsup"}
    if not required_cols.issubset(set(variables_df.columns)):
        raise ValueError(
            f"Variable table must contain columns {required_cols}, "
            f"but got {set(variables_df.columns)}"
        )

    for _, row in variables_df.iterrows():
        var_name = str(row["Variable"]).strip()
        liminf = int(row["liminf"])
        limsup = int(row["limsup"])
        values[var_name] = rng.randint(liminf, limsup)

    return values


def fill_variables(text: str, variable_values: Dict[str, int]) -> str:
    """
    Replace variables like *t_v* or *m_v* with sampled numbers.
    """
    if not text:
        return text

    filled = text
    for var_name, value in variable_values.items():
        filled = filled.replace(var_name, str(value))

    return filled


def fill_variables_in_list(items: List[str], variable_values: Dict[str, int]) -> List[str]:
    """
    Replace variables in every string of a list.
    """
    return [fill_variables(item, variable_values) for item in items]


def fill_variables_in_dataframe(df: pd.DataFrame, variable_values: Dict[str, int]) -> pd.DataFrame:
    """
    Replace variables inside every string cell of a DataFrame.
    Useful for MC_choice.
    """
    if df is None or df.empty:
        return df

    df = df.copy()

    for col in df.columns:
        df[col] = df[col].apply(
            lambda x: fill_variables(str(x), variable_values) if pd.notna(x) else x
        )

    return df


# ============================================================
# 6. LATEX / ANSWER PROCESSING
# ============================================================

def simplify_latex_expression(latex_expr: str) -> str:
    """
    Parse and simplify a LaTeX expression using SymPy.
    If parsing fails, return the original expression.
    """
    latex_expr = clean_math_text(latex_expr)

    try:
        expr = parse_latex(latex_expr)
        simplified = sp.simplify(expr)
        return sp.latex(simplified)
    except Exception:
        # If SymPy cannot parse the expression, just return the original
        return latex_expr


def build_final_answers(answer_list: List[str], unit: str) -> List[str]:
    """
    Take the list of answers, simplify the last expression if possible,
    and append the unit to the final result.

    Example:
        answer_list[-1] = "\\frac{...}{...}"
        unit = "J \\degree C^{-1}"

    Output:
        [
            "...",
            "...",
            "\\frac{...}{...}\\;J \\degree C^{-1}"
        ]
    """
    if not answer_list:
        return []

    output = answer_list.copy()

    # Try simplifying the final answer expression
    final_expr = simplify_latex_expression(output[-1])

    if unit.strip():
        output.append(f"{final_expr}\\;{unit.strip()}")
    else:
        output.append(final_expr)

    return output


def display_latex_lines(lines: List[str], title: Optional[str] = None) -> None:
    """
    Display a list of LaTeX strings nicely in Jupyter.
    """
    if title:
        print(title)

    for line in lines:
        display(Math(line))


# ============================================================
# 7. MAIN PARSER
# ============================================================

def parse_question_file(path: str, seed: Optional[int] = None) -> QuestionData:
    """
    Read a question text file, parse all supported tags,
    generate variable values, fill the question/answers,
    and return a structured QuestionData object.
    """
    with open(path, "r", encoding="utf-8") as file:
        raw_text = file.read()

    raw_text = normalize_text(raw_text)
    tags = extract_all_tags(raw_text)

    rng = random.Random(seed)

    # ------------------------------
    # Parse simple text / number tags
    # ------------------------------
    topic = tags.get("Topic", "")
    question_id = tags.get("Question_ID", "")
    question_type = to_int_or_none(tags.get("Question_type", ""))

    # Backward compatibility:
    # If old "Question" tag exists, use it as English question if Question_en is absent.
    question_en = tags.get("Question_en", "") or tags.get("Question", "")
    question_zh = tags.get("Question_zh", "")

    mark = to_int_or_none(tags.get("Mark", ""))

    hint_en = tags.get("Hint_en", "")
    hint_zh = tags.get("Hint_zh", "")

    unit = tags.get("Unit", "")
    remark = tags.get("Remark", "")

    explanation_en = tags.get("Explanation_en", "")
    explanation_zh = tags.get("Explanation_zh", "")

    image_path = tags.get("Image_path", "")
    image_pos = to_int_or_none(tags.get("Image_pos", ""))

    difficulty = to_int_or_none(tags.get("Difficulty", ""))
    trash = to_int_or_none(tags.get("Trash", ""))

    # ------------------------------
    # Parse table-like blocks
    # ------------------------------
    variables_df = parse_csv_block(tags.get("Variable", ""))
    mc_choice_df = parse_csv_block(tags.get("MC_choice", ""))

    # ------------------------------
    # Parse answer list
    # ------------------------------
    answers_raw = parse_list_block(tags.get("Answer", ""))

    # ------------------------------
    # Generate variable values
    # ------------------------------
    variable_values = sample_variables(variables_df, rng)

    # ------------------------------
    # Fill variables into questions, answers, and MC choices
    # ------------------------------
    question_en_filled = fill_variables(question_en, variable_values)
    question_zh_filled = fill_variables(question_zh, variable_values)

    answers_filled = fill_variables_in_list(answers_raw, variable_values)
    mc_choice_filled = fill_variables_in_dataframe(mc_choice_df, variable_values)

    # Build final answer list with simplified final expression + unit
    answers_with_final_unit = build_final_answers(answers_filled, unit)

    return QuestionData(
        topic=topic,
        question_id=question_id,
        question_type=question_type,

        question_en=question_en_filled,
        question_zh=question_zh_filled,

        mark=mark,
        mc_choice=mc_choice_filled,

        hint_en=hint_en,
        hint_zh=hint_zh,

        variables=variables_df,
        variable_values=variable_values,

        answers_raw=answers_raw,
        answers_filled=answers_filled,
        answers_with_final_unit=answers_with_final_unit,

        unit=unit,
        remark=remark,

        explanation_en=explanation_en,
        explanation_zh=explanation_zh,

        image_path=image_path,
        image_pos=image_pos,

        difficulty=difficulty,
        trash=trash
    )


# ============================================================
# 8. HUMAN-READABLE PRINTER
# ============================================================

def print_question_summary(q: QuestionData) -> None:
    """
    Print all parsed content in a clean, human-readable format.
    """
    print("=" * 70)
    print("QUESTION SUMMARY")
    print("=" * 70)

    print(f"Topic           : {q.topic}")
    print(f"Question ID     : {q.question_id}")
    print(f"Question Type   : {q.question_type}")
    print(f"Mark            : {q.mark}")
    print(f"Difficulty      : {q.difficulty}")
    print(f"Trash           : {q.trash}")
    print(f"Image Path      : {q.image_path}")
    print(f"Image Position  : {q.image_pos}")
    print()

    print("Question (EN):")
    print(q.question_en)
    print()

    print("Question (ZH):")
    print(q.question_zh)
    print()

    print("Hint (EN):")
    print(q.hint_en)
    print()

    print("Hint (ZH):")
    print(q.hint_zh)
    print()

    print("Explanation (EN):")
    print(q.explanation_en)
    print()

    print("Explanation (ZH):")
    print(q.explanation_zh)
    print()

    print("Remark:")
    print(q.remark)
    print()

    print("Unit:")
    print(q.unit)
    print()

    print("Variable Values:")
    print(q.variable_values)
    print()

    print("Variables Table:")
    if q.variables is not None and not q.variables.empty:
        print(q.variables)
    else:
        print("(none)")
    print()

    print("MC Choices:")
    if q.mc_choice is not None and not q.mc_choice.empty:
        print(q.mc_choice)
    else:
        print("(none)")
    print()

    print("Answers (raw):")
    for i, ans in enumerate(q.answers_raw or [], start=1):
        print(f"  {i}. {ans}")
    print()

    print("Answers (filled):")
    for i, ans in enumerate(q.answers_filled or [], start=1):
        print(f"  {i}. {ans}")
    print()

    print("Answers (with final simplified result + unit):")
    for i, ans in enumerate(q.answers_with_final_unit or [], start=1):
        print(f"  {i}. {ans}")
    print()


# ============================================================
# 9. JUPYTER LATEX DISPLAY
# ============================================================

def show_all_latex(q: QuestionData) -> None:
    """
    Display all answer lines in LaTeX format in Jupyter notebook.
    """
    print("LaTeX Answers:")
    if q.answers_with_final_unit:
        for line in q.answers_with_final_unit:
            display(Math(line))
    else:
        print("(No answers found)")


# ============================================================
# 10. SIMPLE WRAPPER FOR OLD USAGE
# ============================================================

def get_question(path: str, seed: Optional[int] = None):
    """
    Backward-compatible function similar to your old version.

    Returns:
        question_en, answers_with_final_unit
    """
    q = parse_question_file(path, seed=seed)
    return q.question_en, q.answers_with_final_unit


In [11]:
q = parse_question_file("A-A-C-21120.question", seed=42)
print_question_summary(q)

QUESTION SUMMARY
Topic           : # extract from file name?
Question ID     : # extract from file name?
Question Type   : None
Mark            : 2
Difficulty      : None
Trash           : None
Image Path      : # Question_ID? file name?
Image Position  : None

Question (EN):
# × °
	A student puts a small metal sphere in the boiling water. After a few minutes, the sphere is quickly transferred to a polystyrene cup containing 56 g of water at a temperature of 20 °C. The cup of water is stirred gently and its highest temperature attained is 23 °C.Given: specific heat capacity of water = 4200 J kg–1 °C–1
	
	Estimate the heat capacity C of the metal sphere.

Question (ZH):
一名學生將一個小金屬球放入沸水中。幾分鐘後，迅速將金屬球轉移到裝有 56 g、20 °C 水的聚苯乙烯杯中。輕輕攪拌杯中的水，使其達到的最高溫度為 23 °C。已知：水的比熱容 = 4200 J kg–1 °C–1

	估算金屬球的熱容量 C。

Hint (EN):


Hint (ZH):


Explanation (EN):
#mainly for MC

Explanation (ZH):
#mainly for MC

Remark:
#AI instruction to specific question

Unit:
J \degree C^{–1}

Variable Values:
{'*t_v*': 23, '*m

In [12]:
show_all_latex(q)

LaTeX Answers:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>